In [22]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import requests

# =========================
# CONFIG
# =========================
STATIONS_FILE = Path(r"D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi/Stations.xlsx")  # sửa lại nếu cần
OUTPUT_DIR = Path("D:/Bussiness_plan/Multimodal_PM25/data/raw/DataSample/Hanoi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPENMETEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

# 11 biến gốc từ Open-Meteo
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "pressure_msl",
    "surface_pressure",
    "precipitation",
    "rain",
    "cloud_cover",
]

COLUMN_RENAME = {
    "time": "datetime_utc",
    "temperature_2m": "temperature_2m_C",
    "relative_humidity_2m": "relative_humidity_2m_pct",
    "dew_point_2m": "dew_point_2m_C",
    "wind_speed_10m": "wind_speed_10m_kmh",
    "wind_direction_10m": "wind_direction_10m_deg",
    "wind_gusts_10m": "wind_gusts_10m_kmh",
    "pressure_msl": "pressure_msl_hPa",
    "surface_pressure": "surface_pressure_hPa",
    "precipitation": "precipitation_mm",
    "rain": "rain_mm",
    "cloud_cover": "cloud_cover_pct",
}

FINAL_COLUMNS = [
    "datetime_utc",
    "location_id",
    "location_name",
    "latitude",
    "longitude",
    "temperature_2m_C",
    "relative_humidity_2m_pct",
    "dew_point_2m_C",
    "wind_speed_10m_kmh",
    "wind_direction_10m_deg",
    "wind_gusts_10m_kmh",
    "pressure_msl_hPa",
    "surface_pressure_hPa",
    "precipitation_mm",
    "rain_mm",
    "cloud_cover_pct",
    "wind_u_10m",
    "wind_v_10m",
    "temp_x_rh",
    "wind_x_rain",
]

def request_json(url, params=None, timeout=60, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, timeout=timeout)
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException as e:
            if attempt == max_retries - 1:
                raise
            sleep_s = min(30, 2 ** attempt)
            print(f"Retry sau {sleep_s}s vì lỗi: {e}")
            time.sleep(sleep_s)

def parse_mixed_date(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        return x
    s = str(x).strip()
    if not s:
        return pd.NaT
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%m/%d/%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except Exception:
            pass
    return pd.to_datetime(s, dayfirst=True, errors="coerce")

def fetch_openmeteo_hourly(latitude, longitude, start_date, end_date):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "GMT",
        "timeformat": "iso8601",
        "cell_selection": "land",
    }

    payload = request_json(OPENMETEO_ARCHIVE_URL, params=params)
    hourly = payload.get("hourly", {})

    if not hourly or "time" not in hourly:
        return pd.DataFrame(), payload

    df = pd.DataFrame(hourly).rename(columns=COLUMN_RENAME)
    df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
    return df, payload

def add_derived_features(df):
    # đổi hướng gió sang radian
    theta = np.deg2rad(df["wind_direction_10m_deg"])

    # Chuẩn khí tượng:
    # u âm khi gió thổi từ đông sang tây, v âm khi gió thổi từ bắc xuống nam
    df["wind_u_10m"] = -df["wind_speed_10m_kmh"] * np.sin(theta)
    df["wind_v_10m"] = -df["wind_speed_10m_kmh"] * np.cos(theta)

    # interactions đơn giản, dễ dùng cho ML baseline
    df["temp_x_rh"] = df["temperature_2m_C"] * df["relative_humidity_2m_pct"]
    df["wind_x_rain"] = df["wind_speed_10m_kmh"] * df["rain_mm"]

    return df

# =========================
# LOAD STATIONS
# =========================
if not STATIONS_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy file: {STATIONS_FILE}")

stations = pd.read_excel(STATIONS_FILE)
stations.columns = [str(c).strip() for c in stations.columns]

required_cols = {
    "Location_id",
    "Location_name",
    "Start Date",
    "End Date",
    "latitude",
    "longitude",
}
missing = required_cols - set(stations.columns)
if missing:
    raise ValueError(f"Thiếu cột trong Stations.xlsx: {missing}")

stations["Start Date"] = stations["Start Date"].apply(parse_mixed_date)
stations["End Date"] = stations["End Date"].apply(parse_mixed_date)

if stations["Start Date"].isna().any() or stations["End Date"].isna().any():
    bad_rows = stations[stations["Start Date"].isna() | stations["End Date"].isna()]
    display(bad_rows)
    raise ValueError("Có station bị lỗi định dạng ngày trong Start Date / End Date.")

if stations["latitude"].isna().any() or stations["longitude"].isna().any():
    bad_rows = stations[stations["latitude"].isna() | stations["longitude"].isna()]
    display(bad_rows)
    raise ValueError("Có station bị thiếu latitude hoặc longitude.")

display(stations)

# =========================
# CRAWL ALL STATIONS
# =========================
all_frames = []

for _, row in stations.iterrows():
    location_id = int(row["Location_id"])
    location_name = str(row["Location_name"]).strip()
    latitude = float(row["latitude"])
    longitude = float(row["longitude"])
    start_date = row["Start Date"].strftime("%Y-%m-%d")
    end_date = row["End Date"].strftime("%Y-%m-%d")

    print(f"\n=== Crawling Open-Meteo cho {location_id} | {location_name} ===")
    print(f"Lat/Lon: {latitude}, {longitude}")
    print(f"Time range: {start_date} -> {end_date}")

    df_weather, payload = fetch_openmeteo_hourly(
        latitude=latitude,
        longitude=longitude,
        start_date=start_date,
        end_date=end_date
    )

    if df_weather.empty:
        print(f"Không có dữ liệu Open-Meteo cho {location_id}")
        continue

    df_weather["location_id"] = location_id
    df_weather["location_name"] = location_name
    df_weather["latitude"] = latitude
    df_weather["longitude"] = longitude

    df_weather = add_derived_features(df_weather)

    for c in FINAL_COLUMNS:
        if c not in df_weather.columns:
            df_weather[c] = pd.NA

    df_weather = df_weather[FINAL_COLUMNS].sort_values("datetime_utc").reset_index(drop=True)

    safe_name = (
        location_name.replace("/", "_")
        .replace("\\", "_")
        .replace(",", "")
        .strip()
    )
    out_csv = OUTPUT_DIR / f"{location_id}_{safe_name}_openmeteo_15features_hourly.csv"
    df_weather.to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"Đã lưu: {out_csv.resolve()} | shape={df_weather.shape}")
    display(df_weather.head(5))

    all_frames.append(df_weather)

# =========================
# COMBINED FILE
# =========================
if all_frames:
    combined = pd.concat(all_frames, ignore_index=True)
    combined = combined.sort_values(["location_id", "datetime_utc"]).reset_index(drop=True)

    combined_csv = OUTPUT_DIR / "openmeteo_15features_hourly_all_stations.csv"
    combined.to_csv(combined_csv, index=False, encoding="utf-8-sig")

    print("\nĐã lưu file gộp:")
    print(combined_csv.resolve())
    print("Combined shape:", combined.shape)
    display(combined.head(10))
else:
    print("Không có station nào crawl thành công.")

,Location_id,Location_name,Start Date,End Date,latitude,longitude
0,2161306,Minh Khai - Bắc Từ Liêm,2024-01-29,2026-04-20,21.0500,105.7400
1,2161292,"Số 46, phố Lưu Quang Vũ",2024-01-29,2026-04-20,21.0152,105.7999



=== Crawling Open-Meteo cho 2161306 | Minh Khai - Bắc Từ Liêm ===
Lat/Lon: 21.05, 105.74
Time range: 2024-01-29 -> 2026-04-20
Đã lưu: D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\2161306_Minh Khai - Bắc Từ Liêm_openmeteo_15features_hourly.csv | shape=(19512, 20)


,datetime_utc,location_id,location_name,latitude,longitude,temperature_2m_C,relative_humidity_2m_pct,dew_point_2m_C,wind_speed_10m_kmh,wind_direction_10m_deg,wind_gusts_10m_kmh,pressure_msl_hPa,surface_pressure_hPa,precipitation_mm,rain_mm,cloud_cover_pct,wind_u_10m,wind_v_10m,temp_x_rh,wind_x_rain
0,2024-01-29 00:00:00+00:00,2161306,Minh Khai - Bắc Từ Liêm,21.05,105.74,11.9,84,9.4,5.2,348,10.4,1024.2,1022.7,0.2,0.2,100,1.081141,-5.086368,999.6,1.04
1,2024-01-29 01:00:00+00:00,2161306,Minh Khai - Bắc Từ Liêm,21.05,105.74,12.4,81,9.2,6.9,28,14.4,1025.0,1023.5,0.0,0.0,100,-3.239354,-6.092338,1004.4,0.00
2,2024-01-29 02:00:00+00:00,2161306,Minh Khai - Bắc Từ Liêm,21.05,105.74,12.9,79,9.4,8.0,5,17.6,1025.7,1024.2,0.0,0.0,100,-0.697246,-7.969558,1019.1,0.00
3,2024-01-29 03:00:00+00:00,2161306,Minh Khai - Bắc Từ Liêm,21.05,105.74,13.4,78,9.7,7.3,11,17.6,1025.4,1023.9,0.0,0.0,100,-1.392906,-7.165878,1045.2,0.00
4,2024-01-29 04:00:00+00:00,2161306,Minh Khai - Bắc Từ Liêm,21.05,105.74,14.8,72,9.8,8.3,5,20.2,1024.3,1022.8,0.0,0.0,100,-0.723393,-8.268416,1065.6,0.00



=== Crawling Open-Meteo cho 2161292 | Số 46, phố Lưu Quang Vũ ===
Lat/Lon: 21.0152, 105.7999
Time range: 2024-01-29 -> 2026-04-20
Đã lưu: D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\2161292_Số 46 phố Lưu Quang Vũ_openmeteo_15features_hourly.csv | shape=(19512, 20)


,datetime_utc,location_id,location_name,latitude,longitude,temperature_2m_C,relative_humidity_2m_pct,dew_point_2m_C,wind_speed_10m_kmh,wind_direction_10m_deg,wind_gusts_10m_kmh,pressure_msl_hPa,surface_pressure_hPa,precipitation_mm,rain_mm,cloud_cover_pct,wind_u_10m,wind_v_10m,temp_x_rh,wind_x_rain
0,2024-01-29 00:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,11.6,84,9.1,8.0,342,15.5,1024.2,1022.4,0.1,0.1,100,2.472136e+00,-7.608452,974.4,0.8
1,2024-01-29 01:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,12.1,82,9.1,7.1,24,15.5,1025.0,1023.2,0.0,0.0,100,-2.887830e+00,-6.486173,992.2,0.0
2,2024-01-29 02:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.1,78,9.4,7.6,360,17.6,1025.6,1023.8,0.0,0.0,100,1.861463e-15,-7.600000,1021.8,0.0
3,2024-01-29 03:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.5,78,9.6,7.2,360,18.0,1025.3,1023.5,0.0,0.0,100,1.763491e-15,-7.200000,1053.0,0.0
4,2024-01-29 04:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,14.6,75,10.1,7.9,360,19.4,1024.3,1022.5,0.0,0.0,100,1.934942e-15,-7.900000,1095.0,0.0



Đã lưu file gộp:
D:\Bussiness_plan\Multimodal_PM25\data\raw\DataSample\Hanoi\openmeteo_15features_hourly_all_stations.csv
Combined shape: (39024, 20)


,datetime_utc,location_id,location_name,latitude,longitude,temperature_2m_C,relative_humidity_2m_pct,dew_point_2m_C,wind_speed_10m_kmh,wind_direction_10m_deg,wind_gusts_10m_kmh,pressure_msl_hPa,surface_pressure_hPa,precipitation_mm,rain_mm,cloud_cover_pct,wind_u_10m,wind_v_10m,temp_x_rh,wind_x_rain
0,2024-01-29 00:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,11.6,84,9.1,8.0,342,15.5,1024.2,1022.4,0.1,0.1,100,2.472136e+00,-7.608452,974.4,0.8
1,2024-01-29 01:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,12.1,82,9.1,7.1,24,15.5,1025.0,1023.2,0.0,0.0,100,-2.887830e+00,-6.486173,992.2,0.0
2,2024-01-29 02:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.1,78,9.4,7.6,360,17.6,1025.6,1023.8,0.0,0.0,100,1.861463e-15,-7.600000,1021.8,0.0
3,2024-01-29 03:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,13.5,78,9.6,7.2,360,18.0,1025.3,1023.5,0.0,0.0,100,1.763491e-15,-7.200000,1053.0,0.0
4,2024-01-29 04:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,14.6,75,10.1,7.9,360,19.4,1024.3,1022.5,0.0,0.0,100,1.934942e-15,-7.900000,1095.0,0.0
5,2024-01-29 05:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,15.6,70,10.3,7.3,351,20.5,1023.1,1021.3,0.0,0.0,100,1.141972e+00,-7.210125,1092.0,0.0
6,2024-01-29 06:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,16.3,68,10.4,6.2,353,19.4,1021.5,1019.7,0.0,0.0,100,7.555899e-01,-6.153786,1108.4,0.0
7,2024-01-29 07:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.5,65,10.9,4.2,340,17.6,1020.5,1018.7,0.0,0.0,100,1.436485e+00,-3.946709,1137.5,0.0
8,2024-01-29 08:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.9,65,11.2,2.2,360,14.8,1019.4,1017.6,0.0,0.0,100,5.388446e-16,-2.200000,1163.5,0.0
9,2024-01-29 09:00:00+00:00,2161292,"Số 46, phố Lưu Quang Vũ",21.0152,105.7999,17.8,66,11.2,4.3,66,11.2,1018.8,1017.0,0.0,0.0,100,-3.928245e+00,-1.748968,1174.8,0.0
